# HTAN — Hyper TransAttUNet
### Medical Image Segmentation Experiments
---

## 0. Setup

In [ ]:
import os
import sys
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torch

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"Project root : {PROJECT_ROOT}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

---
# 1. ISIC-2018 — Skin Lesion Segmentation

## 1.0 Download Data

In [ ]:
from datasets.isic_dataset import download_isic
download_isic()

## 1.1 Verify Data

In [ ]:
TRAIN_IMG_DIR  = Path("/opt/dlami/nvme/HTAN/data/isic/train_images")
TRAIN_MASK_DIR = Path("/opt/dlami/nvme/HTAN/data/isic/train_masks")

imgs  = sorted([f for f in TRAIN_IMG_DIR.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png"}])
masks = sorted([f for f in TRAIN_MASK_DIR.iterdir() if f.suffix.lower() == ".png"])

print(f"Images : {len(imgs)}")
print(f"Masks  : {len(masks)}")
print(f"Match  : {'YES' if len(imgs) == len(masks) else 'NO — CHECK THIS'}")

In [ ]:
# 3 random samples
samples = random.sample(imgs, 3)
fig, axes = plt.subplots(3, 2, figsize=(10, 12))
fig.suptitle("ISIC-2018 Sample Images", fontsize=14, fontweight="bold")

for i, img_path in enumerate(samples):
    mask_path = TRAIN_MASK_DIR / (img_path.stem + "_segmentation.png")
    axes[i, 0].imshow(Image.open(img_path).convert("RGB"))
    axes[i, 0].set_title(f"Input: {img_path.name}")
    axes[i, 0].axis("off")
    axes[i, 1].imshow(Image.open(mask_path).convert("L"), cmap="gray")
    axes[i, 1].set_title(f"Mask: {mask_path.name}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

## 1.2 Sanity Check — Model Forward Pass

In [ ]:
from models.transattunet.TransAttUnet import UNet_Attention_Transformer_Multiscale
from models.baselines.unet import UNet
from models.baselines.doubleunet import DoubleUNet
from models.htan.htan import HTAN_1, HTAN_2, HTAN_1_Hres_only

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
dummy  = torch.randn(2, 3, 256, 256).to(DEVICE)

models_to_check = {
    "unet":             UNet(),
    "doubleunet":       DoubleUNet(),
    "transattunet":     UNet_Attention_Transformer_Multiscale(3, 1),
    "htan_1_n2":        HTAN_1(expansion_n=2),
    "htan_1_n4":        HTAN_1(expansion_n=4),
    "htan_2_n2":        HTAN_2(expansion_n=2),
    "htan_2_n4":        HTAN_2(expansion_n=4),
    "htan_1_hres_only": HTAN_1_Hres_only(expansion_n=4),
}

print(f"{'Model':<22} {'Params':>12} {'Output':>15} {'Status'}")
print("-" * 60)

for name, model in models_to_check.items():
    try:
        model = model.to(DEVICE).eval()
        with torch.no_grad():
            out = model(dummy)
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"{name:<22} {n_params:>12,} {str(tuple(out.shape)):>15}   OK")
    except Exception as e:
        print(f"{name:<22} ERROR — {str(e)[:50]}")
    finally:
        del model
        torch.cuda.empty_cache()

## 1.3 Training

Each cell trains one model independently. Resume is automatic if interrupted.

In [ ]:
# 1.3.1 U-Net
!python train.py --model unet --dataset isic

In [ ]:
# 1.3.2 DoubleU-Net
!python train.py --model doubleunet --dataset isic

In [ ]:
# 1.3.3 TransAttUNet (official code)
!python train.py --model transattunet --dataset isic

In [ ]:
# 1.3.4 HTAN_1 n=2
!python train.py --model htan_1_n2 --dataset isic

In [ ]:
# 1.3.5 HTAN_1 n=4
!python train.py --model htan_1_n4 --dataset isic

In [ ]:
# 1.3.6 HTAN_2 n=2
!python train.py --model htan_2_n2 --dataset isic

In [ ]:
# 1.3.7 HTAN_2 n=4
!python train.py --model htan_2_n4 --dataset isic

In [ ]:
# 1.3.8 HTAN_1 Hres-only (ablation)
!python train.py --model htan_1_hres_only --dataset isic

## 1.4 Evaluate All Models

In [ ]:
!python evaluate.py --model all --dataset isic

## 1.5 Results Table

In [ ]:
RESULTS_DIR = Path("/opt/dlami/nvme/HTAN/results/isic")
SAVES_ROOT  = Path("/opt/dlami/nvme/HTAN/saves")

# Paper numbers — taken directly from Table I (no *)
PAPER_RESULTS = {
    "U-Net†":       {"dice": 67.40, "iou": 54.90, "acc": None,  "rec": 70.80, "pre": None},
    "DoubleU-Net†": {"dice": 89.62, "iou": 82.12, "acc": None,  "rec": 87.80, "pre": 94.59},
    "Swin-UNet†":   {"dice": 89.72, "iou": 82.90, "acc": None,  "rec": 90.32, "pre": 92.04},
    "SegFormer†":   {"dice": 90.24, "iou": 83.60, "acc": None,  "rec": 91.12, "pre": 92.10},
    "MCTrans†":     {"dice": 90.35, "iou": None,  "acc": None,  "rec": None,  "pre": None},
}

MODEL_LABELS = {
    "unet":             "U-Net*",
    "doubleunet":       "DoubleU-Net*",
    "transattunet":     "TransAttUNet_R*",
    "htan_1_n2":        "HTAN_1 n=2 (Ours)",
    "htan_1_n4":        "HTAN_1 n=4 (Ours)",
    "htan_2_n2":        "HTAN_2 n=2 (Ours)",
    "htan_2_n4":        "HTAN_2 n=4 (Ours)",
    "htan_1_hres_only": "HTAN_1 Hres-only (Ours)",
}

OUR_RESULTS = {}
for key, label in MODEL_LABELS.items():
    path = RESULTS_DIR / f"{key}.json"
    if path.exists():
        with open(path) as f:
            data = json.load(f)
        OUR_RESULTS[label] = {k: data[k] for k in ["dice", "iou", "acc", "rec", "pre"]}
    else:
        print(f"Not trained yet: {key}")

def fmt(v):
    return f"{v:.2f}" if v is not None else "—"

ALL = {**PAPER_RESULTS, **OUR_RESULTS}
print(f"\n{'Method':<26} {'Dice':>6} {'IoU':>6} {'ACC':>6} {'REC':>6} {'PRE':>6}")
print("-" * 58)
for name, m in ALL.items():
    print(f"{name:<26} {fmt(m['dice']):>6} {fmt(m['iou']):>6} "
          f"{fmt(m['acc']):>6} {fmt(m['rec']):>6} {fmt(m['pre']):>6}")

## 1.6 Training Curves

In [ ]:
def load_history(model_name):
    path = SAVES_ROOT / model_name / "resume_checkpoint.pth"
    if not path.exists():
        print(f"No checkpoint: {model_name}")
        return None
    return torch.load(path, map_location="cpu")["history"]

def plot_metric(histories, metric, title):
    plt.figure(figsize=(10, 5))
    for name, h in histories.items():
        if h and metric in h:
            plt.plot(h[metric], label=name)
    plt.xlabel("Epoch")
    plt.ylabel(metric.capitalize())
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

MODELS = list(MODEL_LABELS.keys())
histories = {m: load_history(m) for m in MODELS}

plot_metric(histories, "dice",       "1.6.1 Validation Dice")
plot_metric(histories, "iou",        "1.6.2 Validation IoU")
plot_metric(histories, "train_loss", "1.6.3 Training Loss")
plot_metric(histories, "val_loss",   "1.6.4 Validation Loss")

## 1.7 Ablation Table

In [ ]:
ABLATION = [
    ("transattunet",     "—", "—",    "none",  "TransAttUNet_R (baseline)"),
    ("htan_1_hres_only", "1", "4",    "H_res", "HTAN_1 Hres-only"),
    ("htan_1_n2",        "1", "2",    "full",  "HTAN_1 n=2"),
    ("htan_1_n4",        "1", "4",    "full",  "HTAN_1 n=4"),
    ("htan_2_n2",        "2", "2",    "full",  "HTAN_2 n=2"),
    ("htan_2_n4",        "2", "4",    "full",  "HTAN_2 n=4"),
]

print(f"{'Model':<26} {'Blocks':>6} {'n':>4} {'Mappings':>10} {'Dice':>6} {'IoU':>6}")
print("-" * 62)

for key, blocks, n_val, mappings, name in ABLATION:
    path = RESULTS_DIR / f"{key}.json"
    if path.exists():
        with open(path) as f:
            d = json.load(f)
        dice, iou = f"{d['dice']:.2f}", f"{d['iou']:.2f}"
    else:
        dice = iou = "—"
    print(f"{name:<26} {blocks:>6} {n_val:>4} {mappings:>10} {dice:>6} {iou:>6}")

## 1.8 Visual Predictions

In [ ]:
from datasets.isic_dataset import get_loaders

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_best(model, model_name):
    path = SAVES_ROOT / model_name / "best_model.pth"
    if not path.exists():
        print(f"No best model: {model_name}")
        return None
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    return model.to(DEVICE).eval()

_, val_loader = get_loaders(batch_size=4)
imgs, masks   = next(iter(val_loader))
imgs_gpu      = imgs.to(DEVICE)

tan  = load_best(UNet_Attention_Transformer_Multiscale(3, 1), "transattunet")
htan = load_best(HTAN_1(expansion_n=4), "htan_1_n4")

with torch.no_grad():
    pred_tan  = (torch.sigmoid(tan(imgs_gpu))  > 0.5).cpu() if tan  else None
    pred_htan = (torch.sigmoid(htan(imgs_gpu)) > 0.5).cpu() if htan else None

def denorm(t):
    return (t * 0.5 + 0.5).clamp(0, 1)

n_show = min(4, imgs.shape[0])
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
fig.suptitle("1.8 Visual Predictions — ISIC-2018", fontsize=14, fontweight="bold")

for col, title in enumerate(["Input", "Ground Truth", "TransAttUNet", "HTAN_1 n=4"]):
    axes[0, col].set_title(title, fontweight="bold")

for i in range(n_show):
    axes[i, 0].imshow(denorm(imgs[i]).permute(1, 2, 0).numpy())
    axes[i, 1].imshow(masks[i, 0].numpy(), cmap="gray")
    axes[i, 2].imshow(pred_tan[i, 0].numpy()  if pred_tan  is not None else np.zeros((256, 256)), cmap="gray")
    axes[i, 3].imshow(pred_htan[i, 0].numpy() if pred_htan is not None else np.zeros((256, 256)), cmap="gray")
    for ax in axes[i]: ax.axis("off")

plt.tight_layout()
plt.show()

---
# 2. Lung Field Segmentation
*Coming soon — JSRT + Montgomery + NIH datasets*

---
# 3. COVID-19 Pneumonia Segmentation
*Coming soon — Clean-CC-CCII dataset*

---
# 4. Nuclei Segmentation
*Coming soon — 2018 Data Science Bowl dataset*

---
# 5. Gland Segmentation
*Coming soon — GlaS dataset*